# GuardNet — 03. Evaluate Performance

Loads the trained model + preprocessor and evaluates on the **held-out KDDTest+**
split (records never seen during training). Reports accuracy, precision, recall,
F1, a confusion matrix, and the ROC/AUC curve.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, accuracy_score, f1_score)

BASE = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(BASE, 'scripts'))
from threat_detector import load_artifacts, preprocess, COLUMNS

model, scaler, feature_columns, threshold = load_artifacts()
print('Loaded model + preprocessor | decision threshold =', threshold)

In [ ]:
# Load + preprocess the held-out test set, then score it with the model
TEST_PATH = os.path.join(BASE, 'data', 'NSL-KDD Dataset', 'KDDTest+.txt')
df = pd.read_csv(TEST_PATH, header=None, names=COLUMNS)
y_true = (df['label'] != 'normal').astype(int).values

X = preprocess(df, scaler, feature_columns)
probs = model.predict(X, verbose=0).flatten()
y_pred = (probs >= threshold).astype(int)

print('Accuracy:', round(accuracy_score(y_true, y_pred), 4))
print('F1-score:', round(f1_score(y_true, y_pred), 4))
print()
print(classification_report(y_true, y_pred, target_names=['BENIGN', 'ATTACK']))

In [ ]:
# Confusion matrix + ROC curve
plt.style.use('dark_background')
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_true, y_pred)
im = ax[0].imshow(cm, cmap='Blues')
ax[0].set_xticks([0,1]); ax[0].set_yticks([0,1])
ax[0].set_xticklabels(['BENIGN','ATTACK']); ax[0].set_yticklabels(['BENIGN','ATTACK'])
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual'); ax[0].set_title('Confusion Matrix')
for r in range(2):
    for c in range(2):
        ax[0].text(c, r, f'{cm[r,c]:,}', ha='center', va='center',
                   color='white' if cm[r,c] > cm.max()/2 else 'black', fontsize=14)

fpr, tpr, _ = roc_curve(y_true, probs)
ax[1].plot(fpr, tpr, color='#58a6ff', lw=3, label=f'AUC = {auc(fpr, tpr):.4f}')
ax[1].plot([0,1],[0,1], color='#f85149', lw=2, ls='--')
ax[1].set_xlabel('False Positive Rate'); ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC Curve'); ax[1].legend(loc='lower right')
plt.tight_layout(); plt.show()

> For a ready-made shareable image (metric cards + confusion matrix), run
> `python scripts/plot_results.py` — it saves `backend/models/guardnet_results.png`.